In [1]:
# ============================================================
# WOMEN'S CLOTHING SENTIMENT ANALYSIS - DATA PROCESSING PIPELINE
# ============================================================

import pandas as pd
import numpy as np
from sklearn.preprocessing import MinMaxScaler, StandardScaler, LabelEncoder
from sklearn.model_selection import train_test_split

# ============================================================
# STEP 1: DATA COLLECTION - LOAD DATASET
# ============================================================

df = pd.read_csv("Womens Clothing E-Commerce Reviews.csv")
print("Dataset Loaded Successfully!")

# ============================================================
# STEP 2: DATA INSPECTION
# ============================================================

print("\n--- Shape (Rows, Columns) ---")
print(df.shape)

print("\n--- Column Names ---")
print(df.columns.tolist())

print("\n--- Data Types ---")
print(df.dtypes)

print("\n--- First 5 Rows ---")
print(df.head())

print("\n--- Missing Values ---")
print(df.isnull().sum())

print("\n--- Duplicate Records ---")
print("Total Duplicates:", df.duplicated().sum())

print("\n--- Basic Statistics ---")
print(df.describe())

Dataset Loaded Successfully!

--- Shape (Rows, Columns) ---
(23486, 11)

--- Column Names ---
['Unnamed: 0', 'Clothing ID', 'Age', 'Title', 'Review Text', 'Rating', 'Recommended IND', 'Positive Feedback Count', 'Division Name', 'Department Name', 'Class Name']

--- Data Types ---
Unnamed: 0                  int64
Clothing ID                 int64
Age                         int64
Title                      object
Review Text                object
Rating                      int64
Recommended IND             int64
Positive Feedback Count     int64
Division Name              object
Department Name            object
Class Name                 object
dtype: object

--- First 5 Rows ---
   Unnamed: 0  Clothing ID  Age                    Title  \
0           0          767   33                      NaN   
1           1         1080   34                      NaN   
2           2         1077   60  Some major design flaws   
3           3         1049   50         My favorite buy!   
4        

In [2]:
# ============================================================
# STEP 3: DATA CLEANING
# ============================================================

# --- 3a. Drop Unnecessary Column ---
df.drop(columns=['Unnamed: 0'], inplace=True)
print("Unnamed column dropped!")

# --- 3b. Handle Missing Values ---
df['Title'] = df['Title'].fillna("No Title")
df['Review Text'] = df['Review Text'].fillna("No Review")

# Division, Department, Class missing values fix
df['Division Name'] = df['Division Name'].fillna("Unknown")
df['Department Name'] = df['Department Name'].fillna("Unknown")
df['Class Name'] = df['Class Name'].fillna("Unknown")
print("Missing values filled!")

# --- 3c. Remove Duplicates ---
df.drop_duplicates(inplace=True)
df.reset_index(drop=True, inplace=True)
print("Duplicates removed!")

# --- 3d. Verify After Cleaning ---
print("\n--- Missing Values After Cleaning ---")
print(df.isnull().sum())

print("\n--- Duplicates After Cleaning ---")
print("Total Duplicates:", df.duplicated().sum())

print("\n--- Shape After Cleaning ---")
print(df.shape)

Unnamed column dropped!
Missing values filled!
Duplicates removed!

--- Missing Values After Cleaning ---
Clothing ID                0
Age                        0
Title                      0
Review Text                0
Rating                     0
Recommended IND            0
Positive Feedback Count    0
Division Name              0
Department Name            0
Class Name                 0
dtype: int64

--- Duplicates After Cleaning ---
Total Duplicates: 0

--- Shape After Cleaning ---
(23465, 10)


In [3]:
# ============================================================
# STEP 4: DATA TRANSFORMATION
# ============================================================

# --- 4a. Sentiment Label from Rating ---
def get_sentiment(rating):
    if rating in [1, 2]:
        return 'Negative'
    elif rating == 3:
        return 'Neutral'
    else:
        return 'Positive'

df['Sentiment'] = df['Rating'].apply(get_sentiment)
print("--- Sentiment Labels Created ---")
print(df['Sentiment'].value_counts())

# --- 4b. Min-Max Normalization on Age ---
scaler_minmax = MinMaxScaler()
df['Age'] = scaler_minmax.fit_transform(df[['Age']])
print("\n--- Age After Min-Max Normalization ---")
print(df['Age'].head())

# --- 4c. Z-Score Normalization on Positive Feedback Count ---
scaler_zscore = StandardScaler()
df['Positive Feedback Count'] = scaler_zscore.fit_transform(df[['Positive Feedback Count']])
print("\n--- Positive Feedback Count After Z-Score ---")
print(df['Positive Feedback Count'].head())

# ============================================================
# STEP 5: ENCODING CATEGORICAL COLUMNS
# ============================================================

le = LabelEncoder()

df['Division Name'] = le.fit_transform(df['Division Name'])
print("\n--- Division Name Encoded ---")
print(df['Division Name'].value_counts())

df['Department Name'] = le.fit_transform(df['Department Name'])
print("\n--- Department Name Encoded ---")
print(df['Department Name'].value_counts())

df['Class Name'] = le.fit_transform(df['Class Name'])
print("\n--- Class Name Encoded ---")
print(df['Class Name'].value_counts())

print("\n--- Shape After Encoding ---")
print(df.shape)

print("\n--- Sample Data After Transformation ---")
print(df.head())

--- Sentiment Labels Created ---
Sentiment
Positive    18187
Neutral      2871
Negative     2407
Name: count, dtype: int64

--- Age After Min-Max Normalization ---
0    0.185185
1    0.197531
2    0.518519
3    0.395062
4    0.358025
Name: Age, dtype: float64

--- Positive Feedback Count After Z-Score ---
0   -0.444977
1    0.256270
2   -0.444977
3   -0.444977
4    0.606893
Name: Positive Feedback Count, dtype: float64

--- Division Name Encoded ---
Division Name
0    13839
1     8110
2     1502
3       14
Name: count, dtype: int64

--- Department Name Encoded ---
Department Name
4    10455
1     6312
0     3798
2     1735
3     1032
5      119
6       14
Name: count, dtype: int64

--- Class Name Encoded ---
Class Name
3     6312
8     4835
0     3093
17    1428
13    1388
7     1146
4     1099
15     945
6      704
11     691
18     350
12     328
14     317
16     228
10     165
5      154
9      146
19     119
20      14
1        2
2        1
Name: count, dtype: int64

--- Shape Aft

In [4]:
# ============================================================
# STEP 6: FEATURE ENGINEERING
# ============================================================

# --- Discretization - Age Groups ---
def age_group(age):
    if age <= 0.30:
        return 'Young'
    elif age <= 0.60:
        return 'Middle'
    else:
        return 'Senior'

df['Age_Group'] = df['Age'].apply(age_group)
print("--- Age Groups Created ---")
print(df['Age_Group'].value_counts())

--- Age Groups Created ---
Age_Group
Young     12627
Middle     9955
Senior      883
Name: count, dtype: int64


In [7]:
# ============================================================
# STEP 7: DATA SPLITTING
# ============================================================

# --- X (Input) aur y (Target) alag karo ---
X = df['Review Text']
y = df['Sentiment']

print("--- Input (X) Shape ---")
print(X.shape)

print("\n--- Target (y) Shape ---")
print(y.shape)

print("\n--- Target Distribution ---")
print(y.value_counts())

# --- Pehle 70% Train, 30% baqi ---
X_train, X_temp, y_train, y_temp = train_test_split(
    X, y, 
    test_size=0.30, 
    random_state=42,
    stratify=y
)

# --- 30% ko aadha aadha --- 15% Test, 15% Validation ---
X_test, X_val, y_test, y_val = train_test_split(
    X_temp, y_temp, 
    test_size=0.50, 
    random_state=42,
    stratify=y_temp
)

print("\n--- Split Sizes ---")
print("Total Dataset :", len(df))
print("Training Set  :", len(X_train), f"({len(X_train)/len(df)*100:.1f}%)")
print("Test Set      :", len(X_test), f"({len(X_test)/len(df)*100:.1f}%)")
print("Validation Set:", len(X_val), f"({len(X_val)/len(df)*100:.1f}%)")

# ============================================================
# STEP 7: SAVE FINAL PROCESSED DATASET
# ============================================================

df.to_csv("Cleaned_Reviews_File.csv", index=False)
print("\nFinal dataset saved as Cleaned_Reviews_File.csv")
print("\n--- Pipeline Complete! ---")

--- Input (X) Shape ---
(23465,)

--- Target (y) Shape ---
(23465,)

--- Target Distribution ---
Sentiment
Positive    18187
Neutral      2871
Negative     2407
Name: count, dtype: int64

--- Split Sizes ---
Total Dataset : 23465
Training Set  : 16425 (70.0%)
Test Set      : 3520 (15.0%)
Validation Set: 3520 (15.0%)

Final dataset saved as Cleaned_Reviews_File.csv

--- Pipeline Complete! ---
